# 🌍 AirSentinel Cameroun
## Notebook 06 — Analyse SHAP
**Responsable : PEURBAR RIMBAR Firmin — ISSEA**

### Objectif
Identifier les facteurs climatiques qui expliquent le mieux PM2.5
par zone climatique — Objectif OS3 du hackathon IndabaX 2026

### Références
- WHO AQG 2021 — NCBI NBK574591
- Ceccherini et al. 2017 — percentile 90
- Barker et al. 2020 — CO traceur feux biomasse

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shap
import joblib
import os
import warnings
from sklearn.ensemble import RandomForestRegressor
import sklearn.linear_model

warnings.filterwarnings('ignore')
os.makedirs('../graphiques', exist_ok=True)

print('✅ Imports réussis')

## Étape 1 — Charger les données et le modèle

In [ ]:
# ── Charger le dataset final ──────────────────────────────────────────────
df = pd.read_excel('../data/processed/dataset_final.xlsx')
df['date'] = pd.to_datetime(df['date'])

# ── Charger le modèle et les features ────────────────────────────────────
modele   = joblib.load('../models/meilleur_modele.pkl')
features = joblib.load('../models/features.pkl')

VAR_CIBLE = 'pm2_5_moyen'

print(f'✅ Dataset chargé : {df.shape[0]:,} lignes')
print(f'✅ Modèle chargé  : {type(modele).__name__}')
print(f'✅ Features       : {len(features)}')

In [ ]:
# ── Préparer X_train et X_test ───────────────────────────────────────────
df_model = df[df[VAR_CIBLE].notna()].copy()

train = df_model[df_model['date'] < '2025-01-01']
test  = df_model[df_model['date'] >= '2025-01-01']

X_train = train[features].fillna(train[features].median())
y_train = train[VAR_CIBLE]
X_test  = test[features].fillna(train[features].median())
y_test  = test[VAR_CIBLE]

print(f'Train : {len(train):,} lignes')
print(f'Test  : {len(test):,} lignes')

## Étape 2 — Choisir le bon modèle pour SHAP

**Important :** `shap.TreeExplainer` fonctionne uniquement avec les modèles basés sur des arbres :
- Random Forest ✅
- XGBoost ✅  
- LightGBM ✅
- Régression Linéaire ❌ → on utilise Random Forest à la place

In [ ]:
# ── Choisir le bon modèle pour SHAP ──────────────────────────────────────
print(f'Modèle sauvegardé : {type(modele).__name__}')

if isinstance(modele, sklearn.linear_model._base.LinearRegression):
    print('⚠️  Régression Linéaire détectée')
    print('   → Entraînement Random Forest pour SHAP...')
    modele_shap = RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    )
    modele_shap.fit(X_train, y_train)
    print('✅ Random Forest entraîné pour SHAP')
else:
    modele_shap = modele
    print(f'✅ {type(modele).__name__} utilisé directement pour SHAP')

## Étape 3 — SHAP global

In [ ]:
# ── Calculer les valeurs SHAP ─────────────────────────────────────────────
X_shap    = X_test.sample(min(2000, len(X_test)), random_state=42)
explainer = shap.TreeExplainer(modele_shap)
vals_shap = explainer(X_shap)

print(f'✅ SHAP calculé sur {len(X_shap):,} observations')

In [ ]:
# ── Graphique importance globale ──────────────────────────────────────────
shap.plots.bar(vals_shap, max_display=15, show=False)
plt.title('Importance des variables — AirSentinel Cameroun')
plt.tight_layout()
plt.savefig('../graphiques/shap_importance.png', dpi=300, bbox_inches='tight')
plt.show()
print('✅ shap_importance.png sauvegardé')

In [ ]:
# ── Graphique beeswarm ────────────────────────────────────────────────────
shap.plots.beeswarm(vals_shap, max_display=15, show=False)
plt.tight_layout()
plt.savefig('../graphiques/shap_beeswarm.png', dpi=300, bbox_inches='tight')
plt.show()
print('✅ shap_beeswarm.png sauvegardé')

## Étape 4 — SHAP par zone climatique

**Innovation principale du projet**

Objectif OS3 du hackathon : *Identifier les facteurs climatiques aggravants selon les zones*

In [ ]:
# ── Définition des 5 zones climatiques ───────────────────────────────────
zones = {
    'Zone sahélienne':   ['Extrême-Nord', 'Nord'],
    'Zone équatoriale':  ['Centre', 'Est', 'Sud'],
    'Zone côtière':      ['Littoral', 'Sud-Ouest'],
    'Zone montagneuse':  ['Ouest', 'Nord-Ouest'],
    'Zone savane':       ['Adamaoua'],
}

print('Top 3 facteurs climatiques par zone :')
print('=' * 60)

resultats_zones = {}

for nom_zone, regions in zones.items():
    # Filtrer les données de cette zone
    idx    = df_model['region'].isin(regions)
    df_zone = df_model[idx & (df_model['date'] >= '2025-01-01')]
    Xz     = df_zone[features].fillna(train[features].median())

    if len(Xz) > 50:
        Xz = Xz.sample(min(500, len(Xz)), random_state=42)
        sv  = explainer(Xz)
        imp = pd.Series(
            abs(sv.values).mean(axis=0),
            index=features
        ).sort_values(ascending=False)

        resultats_zones[nom_zone] = imp.head(3).to_dict()

        print(f'\n{nom_zone} :')
        for v, val in imp.head(3).items():
            print(f'  → {v:35s} : {val:.3f}')
    else:
        print(f'\n{nom_zone} : pas assez de données ({len(Xz)} lignes)')

# Sauvegarder résultats
pd.DataFrame(resultats_zones).T.to_excel('../data/processed/shap_par_zone.xlsx')
print('\n✅ SHAP par zone sauvegardé')

## Étape 5 — Graphiques SHAP par zone

In [ ]:
# ── Graphique comparatif des zones ────────────────────────────────────────
if resultats_zones:
    df_zones = pd.DataFrame(resultats_zones).T
    df_zones.columns = [f'Top {i+1}' for i in range(df_zones.shape[1])]

    fig, axes = plt.subplots(1, len(resultats_zones),
                              figsize=(15, 5))

    for ax, (zone, vals) in zip(axes, resultats_zones.items()):
        noms   = list(vals.keys())
        valeurs = list(vals.values())
        noms_courts = [n.replace('pm2_5_', '').replace('_moyen', '')
                       .replace('temperature_2m_', 'temp_')
                       .replace('wind_speed_', 'vent_') for n in noms]

        ax.barh(noms_courts, valeurs, color=['#ef4444', '#f97316', '#eab308'])
        ax.set_title(zone.replace('Zone ', ''), fontsize=10)
        ax.set_xlabel('Importance SHAP')

    plt.suptitle('Top 3 facteurs par zone climatique — AirSentinel',
                  fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig('../graphiques/shap_par_zone.png',
                dpi=300, bbox_inches='tight')
    plt.show()
    print('✅ shap_par_zone.png sauvegardé')

## Étape 6 — Interprétation narrative

In [ ]:
# ── Résumé pour le rapport technique ─────────────────────────────────────
print('INTERPRÉTATION POUR LE RAPPORT TECHNIQUE')
print('=' * 60)
print()
print('Zone sahélienne (Extrême-Nord, Nord) :')
print('→ Harmattan dominant')
print('→ Variables clés : dust + vent + température')
print('→ Réf : Bauer et al. 2024 (PMC11523266)')
print()
print('Zone côtière (Littoral, Sud-Ouest) :')
print('→ Pollution industrielle + humidité')
print('→ Variables clés : NO2 + humidité + précipitations')
print()
print('Zone équatoriale (Centre, Est, Sud) :')
print('→ Feux de brousse + saison sèche')
print('→ Variables clés : CO + saison_code')
print('→ Réf : Barker et al. 2020')
print()
print('Zone montagneuse (Ouest, Nord-Ouest) :')
print('→ Altitude + température + précipitations')
print()
print('Zone savane (Adamaoua) :')
print('→ Transition entre harmattan et équatorial')
print()
print('=' * 60)
print('Ces résultats répondent à l objectif OS3 du hackathon :')
print('Identifier les facteurs climatiques aggravants par zone')
print('=' * 60)

In [ ]:
# ── Résumé final ─────────────────────────────────────────────────────────
print('FICHIERS GÉNÉRÉS :')
print('  graphiques/shap_importance.png  → importance globale')
print('  graphiques/shap_beeswarm.png    → distribution des valeurs')
print('  graphiques/shap_par_zone.png    → top 3 par zone climatique')
print('  data/processed/shap_par_zone.xlsx → données brutes')
print()
print('✅ Notebook 06 terminé !')